# Competitor Price Distribution Model

This notebook trains a **monotonic multi-quantile XGBoost regressor** to model
the full distribution of competitor prices for a given insurance quote.

## Why quantile regression?

Standard regression predicts a single expected value. In a competitive pricing
setting we want to understand the *distribution* of what the market charges —
not just the mean. By simultaneously estimating 19 quantiles (q5 → q95) we get:

- The **range** of prices a competitor might quote for a given risk profile
- The **median** market price (q50)
- The **tails** — how aggressive (q5) or conservative (q95) competitors can be

## Data

Each row is a `(quote, competitor)` pair. Features describe the insured risk;
the target is the price that competitor quoted for that quote.
There are 5 competitors, so each quote appears 5 times with different prices.

## Approach

1. Split by `quote_id` (not by row) to prevent leakage across quote-competitor pairs
2. Build a `FeatureUnion` preprocessing pipeline over three feature groups
3. Tune XGBoost hyperparameters with Optuna (Bayesian optimisation)
4. Train the final model and evaluate quantile calibration on the held-out set

In [1]:
import sys
sys.path.insert(0, '../')

import random
import os

import pandas as pd
import numpy as np
import joblib
import optuna

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer
from category_encoders import TargetEncoder

from transformers import FeatureSelectorValues, NeverSeenToNanEncoder, MonotonicQuantileXGBRegressor

pd.set_option('display.max_columns', 100)

## 1. Load Data

In [2]:
df = pd.read_csv('../data/train_data.csv')

print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Unique quotes: {df['quote_id'].nunique():,}")
print(f"Competitors:   {df['competitor_id'].nunique()} ({sorted(df['competitor_id'].unique())})")
print(f"\ntarget_price:")
print(df['target_price'].describe().round(2))
df.head(10)

Dataset: 5,000 rows, 33 columns
Unique quotes: 1,000
Competitors:   5 (['comp_1', 'comp_2', 'comp_3', 'comp_4', 'comp_5'])

target_price:
count     5000.00
mean     10601.68
std       3964.70
min       2416.46
25%       7752.74
50%       9921.49
75%      12760.13
max      35131.19
Name: target_price, dtype: float64


,quote_id,competitor_id,num_feature_1,num_feature_2,num_feature_3,num_feature_4,num_feature_5,num_feature_6,num_feature_7,num_feature_8,num_feature_9,num_feature_10,num_feature_11,num_feature_12,geo_feature_1,geo_feature_2,geo_feature_3,geo_feature_4,geo_feature_5,geo_feature_6,geo_feature_7,geo_feature_8,geo_feature_9,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,target_price
0,Q00000,comp_1,40,1,0.285,2,2,260.0,1.4,16,15,66.9,16,9.4,-2.3967,0.4018,2.3751,1.1987,-0.3330,2.6881,-2.1509,2.0266,1.2877,region_H,special,tariff_6,domestic,electric,electric,DOHC,gen_4,M,15745.45
1,Q00000,comp_2,40,1,0.285,2,2,260.0,1.4,16,15,66.9,16,9.4,-2.3967,0.4018,2.3751,1.1987,-0.3330,2.6881,-2.1509,2.0266,1.2877,region_H,special,tariff_6,domestic,electric,electric,DOHC,gen_4,M,12763.78
2,Q00000,comp_3,40,1,0.285,2,2,260.0,1.4,16,15,66.9,16,9.4,-2.3967,0.4018,2.3751,1.1987,-0.3330,2.6881,-2.1509,2.0266,1.2877,region_H,special,tariff_6,domestic,electric,electric,DOHC,gen_4,M,12436.01
3,Q00000,comp_4,40,1,0.285,2,2,260.0,1.4,16,15,66.9,16,9.4,-2.3967,0.4018,2.3751,1.1987,-0.3330,2.6881,-2.1509,2.0266,1.2877,region_H,special,tariff_6,domestic,electric,electric,DOHC,gen_4,M,17675.36
4,Q00000,comp_5,40,1,0.285,2,2,260.0,1.4,16,15,66.9,16,9.4,-2.3967,0.4018,2.3751,1.1987,-0.3330,2.6881,-2.1509,2.0266,1.2877,region_H,special,tariff_6,domestic,electric,electric,DOHC,gen_4,M,14568.05
5,Q00001,comp_1,61,4,0.103,4,4,129.0,4.0,19,20,64.9,4,8.7,0.7974,-0.0353,-1.8007,-0.3628,-0.3118,-0.3733,-0.3736,1.2336,-0.1490,region_B,standard,tariff_2,domestic,diesel,hydraulic,SOHC,gen_2,F,10443.88
6,Q00001,comp_2,61,4,0.103,4,4,129.0,4.0,19,20,64.9,4,8.7,0.7974,-0.0353,-1.8007,-0.3628,-0.3118,-0.3733,-0.3736,1.2336,-0.1490,region_B,standard,tariff_2,domestic,diesel,hydraulic,SOHC,gen_2,F,8279.26
7,Q00001,comp_3,61,4,0.103,4,4,129.0,4.0,19,20,64.9,4,8.7,0.7974,-0.0353,-1.8007,-0.3628,-0.3118,-0.3733,-0.3736,1.2336,-0.1490,region_B,standard,tariff_2,domestic,diesel,hydraulic,SOHC,gen_2,F,9680.84
8,Q00001,comp_4,61,4,0.103,4,4,129.0,4.0,19,20,64.9,4,8.7,0.7974,-0.0353,-1.8007,-0.3628,-0.3118,-0.3733,-0.3736,1.2336,-0.1490,region_B,standard,tariff_2,domestic,diesel,hydraulic,SOHC,gen_2,F,18738.50
9,Q00001,comp_5,61,4,0.103,4,4,129.0,4.0,19,20,64.9,4,8.7,0.7974,-0.0353,-1.8007,-0.3628,-0.3118,-0.3733,-0.3736,1.2336,-0.1490,region_B,standard,tariff_2,domestic,diesel,hydraulic,SOHC,gen_2,F,8164.84


## 2. Train / Test Split

We split by **`quote_id`**, not by row. Since each quote appears once per
competitor, a row-level split would leak information: the model could learn
the price for quote Q001 from comp_1's row and then evaluate on Q001 from
comp_2's row — artificially inflating test performance.

In [3]:
def splitter(df, frac=0.7, seed=42):
    quote_ids = df['quote_id'].unique().tolist()
    rng = random.Random(seed)
    train_ids = set(rng.sample(quote_ids, int(len(quote_ids) * frac)))
    train = df[df['quote_id'].isin(train_ids)]
    test  = df[~df['quote_id'].isin(train_ids)]
    return train, test


X_train, X_test = splitter(df)
y_train = X_train['target_price']
y_test  = X_test['target_price']

print(f"Train: {X_train.shape[0]:,} rows ({X_train['quote_id'].nunique():,} quotes)")
print(f"Test:  {X_test.shape[0]:,} rows ({X_test['quote_id'].nunique():,} quotes)")

Train: 3,500 rows (700 quotes)
Test:  1,500 rows (300 quotes)


## 3. Feature Engineering Pipeline

We use a `FeatureUnion` to process three feature groups in parallel and
concatenate their outputs before passing them to the model:

| Group | Transformer | Rationale |
|---|---|---|
| `num_feature_*` | Median imputation | Continuous numeric features |
| `geo_feature_*` | Median imputation | Standardised continuous indicators |
| `competitor_id`, `cat_feature_1`, `cat_feature_3` | Target encoding | Moderate-to-high cardinality — supervised mapping to mean price preserves ordinal signal for tree splits |
| `cat_feature_2`, `cat_feature_4..9` | One-hot encoding | Low cardinality (≤4 levels) — no ordinal relationship assumed |

`NeverSeenToNanEncoder` guards each categorical pipeline: any category value
not seen during training is set to NaN before encoding, preventing silent
errors when the model is applied to new data.

In [4]:
alphas = np.arange(0.05, 1.0, 0.05)  # 19 quantile levels: q5, q10, ..., q95

# ── Numeric features ──────────────────────────────────────────────────────────
numerical_pipeline = Pipeline(steps=[
    ('selector', FeatureSelectorValues([f'num_feature_{i}' for i in range(1, 13)])),
    ('imputer',  SimpleImputer(strategy='median')),
])

geo_pipeline = Pipeline(steps=[
    ('selector', FeatureSelectorValues([f'geo_feature_{i}' for i in range(1, 10)])),
    ('imputer',  SimpleImputer(strategy='median')),
])

# ── Categorical encoding helpers ──────────────────────────────────────────────
def te_pipeline(col):
    """Target-encode a single categorical column."""
    return Pipeline(steps=[
        ('handle_unknown', NeverSeenToNanEncoder(col=col)),
        ('imputer',        SimpleImputer(strategy='most_frequent')),
        ('encoder',        TargetEncoder()),
    ])

def ohe_pipeline(col):
    """One-hot encode a single categorical column."""
    return Pipeline(steps=[
        ('handle_unknown', NeverSeenToNanEncoder(col=col)),
        ('imputer',        SimpleImputer(strategy='most_frequent')),
        ('encoder',        OneHotEncoder(handle_unknown='ignore')),
    ])

# Target-encoded: moderate-to-high cardinality
te_cols  = ['competitor_id', 'cat_feature_1', 'cat_feature_3']
# One-hot encoded: low cardinality
ohe_cols = ['cat_feature_2', 'cat_feature_4', 'cat_feature_5',
            'cat_feature_6', 'cat_feature_7', 'cat_feature_8', 'cat_feature_9']

# ── Assemble ──────────────────────────────────────────────────────────────────
feature_union = FeatureUnion(transformer_list=
    [('numerical', numerical_pipeline), ('geo', geo_pipeline)]
    + [(col, te_pipeline(col))  for col in te_cols]
    + [(col, ohe_pipeline(col)) for col in ohe_cols]
)

full_pipeline = Pipeline([
    ('features', feature_union),
    ('model',    MonotonicQuantileXGBRegressor(quantile_alpha=alphas)),
])

## 4. Hyperparameter Search with Optuna

We tune XGBoost hyperparameters using **Bayesian optimisation** (Optuna's
Tree-structured Parzen Estimator). The objective is to maximise **mean quantile
calibration** across all 19 quantile levels:

$$\text{mqloss} = \frac{1}{|\alpha|} \sum_{\alpha} \left| \hat{F}(\alpha) - \alpha \right|$$

where $\hat{F}(\alpha)$ is the fraction of test points below the predicted
$\alpha$-quantile. A perfectly calibrated model gives 0; worst possible is 0.5.
We evaluate via 3-fold cross-validation.

In [5]:
def mqloss(y_true, y_pred, alphas):
    """Mean quantile calibration loss (negated so higher = better, for sklearn's maximise convention)."""
    error = sum(
        abs((y_true - y_pred[:, i] < 0).mean() - alpha)
        for i, alpha in enumerate(alphas)
    )
    return -(error / len(alphas))


def objective(trial):
    params = {
        'model__objective':        'reg:quantileerror',
        'model__tree_method':      'hist',
        'model__learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__n_estimators':     trial.suggest_int('n_estimators', 50, 200),
        'model__max_depth':        trial.suggest_int('max_depth', 3, 8),
        'model__min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'model__subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'model__colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }
    full_pipeline.set_params(**params)
    scores = cross_val_score(
        full_pipeline, X_train, y_train,
        cv=3, n_jobs=-1,
        scoring=make_scorer(mqloss, alphas=alphas, greater_is_better=True),
    )
    return scores.mean()


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, gc_after_trial=True)

print(f"Best calibration loss : {study.best_value:.4f}  (0 = perfect, -0.5 = worst)")
print(f"Best params           : {study.best_params}")

Best calibration loss : -0.0204  (0 = perfect, -0.5 = worst)
Best params           : {'learning_rate': 0.27998382012025774, 'n_estimators': 167, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.611630723028943, 'colsample_bytree': 0.7815654051501087}


## 5. Train Final Model

Re-train on the full training set with the best hyperparameters, then evaluate
quantile calibration on the held-out test set.

In [6]:
best_params = {'model__' + k: v for k, v in study.best_params.items()}
best_params.update({
    'model__objective':   'reg:quantileerror',
    'model__tree_method': 'hist',
})

full_pipeline.set_params(**best_params)
full_pipeline.fit(X_train, y_train)

# Evaluate on test set
test_preds = full_pipeline.predict(X_test)
test_score = mqloss(y_test.values, test_preds, alphas)
print(f"Test mqloss: {test_score:.4f}  (0 = perfect calibration, -0.5 = worst)")

# Per-quantile coverage check
coverages = [(y_test.values < test_preds[:, i]).mean() for i in range(len(alphas))]
print("\nQuantile calibration (nominal vs actual coverage):")
for alpha, cov in zip(alphas, coverages):
    print(f"  q{alpha:.0%}  nominal={alpha:.2f}  actual={cov:.2f}  diff={cov-alpha:+.2f}")

Test mqloss: -0.0104  (0 = perfect calibration, -0.5 = worst)

Quantile calibration (nominal vs actual coverage):
  q5%  nominal=0.05  actual=0.06  diff=+0.01
  q10%  nominal=0.10  actual=0.10  diff=+0.00
  q15%  nominal=0.15  actual=0.16  diff=+0.01
  q20%  nominal=0.20  actual=0.22  diff=+0.02
  q25%  nominal=0.25  actual=0.27  diff=+0.02
  q30%  nominal=0.30  actual=0.32  diff=+0.02
  q35%  nominal=0.35  actual=0.37  diff=+0.02
  q40%  nominal=0.40  actual=0.41  diff=+0.01
  q45%  nominal=0.45  actual=0.46  diff=+0.01
  q50%  nominal=0.50  actual=0.50  diff=+0.00
  q55%  nominal=0.55  actual=0.56  diff=+0.01
  q60%  nominal=0.60  actual=0.61  diff=+0.01
  q65%  nominal=0.65  actual=0.65  diff=+0.00
  q70%  nominal=0.70  actual=0.71  diff=+0.01
  q75%  nominal=0.75  actual=0.75  diff=+0.00
  q80%  nominal=0.80  actual=0.81  diff=+0.01
  q85%  nominal=0.85  actual=0.84  diff=-0.01
  q90%  nominal=0.90  actual=0.88  diff=-0.02
  q95%  nominal=0.95  actual=0.93  diff=-0.02


In [7]:
os.makedirs('../models', exist_ok=True)
joblib.dump(full_pipeline, '../models/model.pkl')
print("Model saved to ../models/model.pkl")

Model saved to ../models/model.pkl
